# 🌟 Star Classification — Complete Machine Learning Project
## SDSS (Sloan Digital Sky Survey) Dataset

**Objective:** Classify astronomical objects as **GALAXY**, **STAR**, or **QSO** (Quasar)  
**Dataset:** 100,000 rows × 18 columns  
**Algorithms Used:** 12 ML algorithms (Linear → Ensemble → Neural Network)

---
### 📋 Table of Contents
1. Understanding the Dataset  
2. Descriptive Statistics  
3. Exploratory Data Analysis (EDA)  
4. Data Cleaning  
5. Outlier Handling  
6. Data Preprocessing  
7. Feature Engineering  
8. Train-Test Split & Scaling  
9. Machine Learning Models (All 12)  
10. Model Comparison & Conclusion


## Step 1: Import Libraries

In [ ]:
# ─────────────────────────────────────────────────
# IMPORT ALL REQUIRED LIBRARIES
# ─────────────────────────────────────────────────
import numpy as np                          # For numerical operations
import pandas as pd                         # For data manipulation
import matplotlib.pyplot as plt             # For plotting
import seaborn as sns                       # For beautiful plots
import warnings
warnings.filterwarnings('ignore')           # Suppress unnecessary warnings

# Scikit-learn: data prep
from sklearn.model_selection import train_test_split  # Split data
from sklearn.preprocessing import LabelEncoder, RobustScaler  # Encode + scale
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# Individual ML models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Ensemble models
from sklearn.ensemble import (RandomForestClassifier,
                               ExtraTreesClassifier,
                               AdaBoostClassifier,
                               BaggingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Plot settings
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
print("✅ All libraries imported successfully!")


## Step 2: Load & Understand the Dataset

In [ ]:
# ─────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────
df = pd.read_csv('star_classification.csv')

# Basic info
print(f"Shape       : {df.shape}")
print(f"Rows        : {df.shape[0]:,}")
print(f"Columns     : {df.shape[1]}")
print(f"\nColumn Names: {df.columns.tolist()}")


In [ ]:
# ─────────────────────────────────────────────────
# WHAT DOES EACH COLUMN MEAN?
# ─────────────────────────────────────────────────
feature_info = {
    'obj_ID'     : 'Unique object ID — just an identifier, not useful for ML',
    'alpha'      : 'Right Ascension — sky coordinate (East-West position)',
    'delta'      : 'Declination — sky coordinate (North-South position)',
    'u'          : 'Ultraviolet filter brightness (photometric band)',
    'g'          : 'Green filter brightness (photometric band)',
    'r'          : 'Red filter brightness (photometric band)',
    'i'          : 'Near-Infrared filter brightness (photometric band)',
    'z'          : 'Infrared filter brightness (photometric band)',
    'run_ID'     : 'Scan run number — administrative ID',
    'rerun_ID'   : 'Reprocessing spec — administrative ID',
    'cam_col'    : 'Camera column number — administrative ID',
    'field_ID'   : 'Field number — administrative ID',
    'spec_obj_ID': 'Spectroscopic object ID — administrative ID',
    'class'      : '🎯 TARGET — GALAXY, STAR, or QSO (what we want to predict)',
    'redshift'   : 'How fast object moves away from us (very important feature!)',
    'plate'      : 'Plate number — administrative ID',
    'MJD'        : 'Modified Julian Date — when data was collected',
    'fiber_ID'   : 'Fiber ID number — administrative ID',
}
print("Feature Descriptions:\n" + "-"*60)
for k, v in feature_info.items():
    print(f"  {k:<15}: {v}")


In [ ]:
# ─────────────────────────────────────────────────
# FIRST LOOK AT THE DATA
# ─────────────────────────────────────────────────
print("First 5 rows of the dataset:")
df.head()


In [ ]:
# Data types and null count
print("Data Types and Non-Null Count:")
df.info()


In [ ]:
# ─────────────────────────────────────────────────
# CHECK TARGET VARIABLE
# ─────────────────────────────────────────────────
print("Class Distribution (Target Variable):")
print(df['class'].value_counts())
print()
print("Class Percentages:")
print(df['class'].value_counts(normalize=True).round(3) * 100)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
COLORS = {'GALAXY': '#3498db', 'STAR': '#f39c12', 'QSO': '#e74c3c'}
counts = df['class'].value_counts()

axes[0].bar(counts.index, counts.values,
            color=[COLORS[c] for c in counts.index], edgecolor='black', width=0.5)
axes[0].set_title('Count of Each Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (c, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[COLORS[c] for c in counts.index],
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proportion of Each Class', fontsize=13, fontweight='bold')

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# WHY THIS MATTERS:
# GALAXY: 59,445 (59.4%) — majority class
# STAR:   21,594 (21.6%) — minority
# QSO:    18,961 (19.0%) — minority
# → Slight imbalance but manageable. No special resampling needed.


## Step 3: Descriptive Statistics

In [ ]:
# ─────────────────────────────────────────────────
# BASIC STATISTICS
# ─────────────────────────────────────────────────
print("Statistical Summary of Key Features:\n")
df[['u','g','r','i','z','redshift','alpha','delta']].describe().round(2)


In [ ]:
# ─────────────────────────────────────────────────
# MISSING VALUES CHECK
# ─────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing Values per Column:")
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

# DUPLICATE CHECK
dupes = df.duplicated().sum()
print(f"\nDuplicate Rows: {dupes}")
print("✅ No duplicates!" if dupes == 0 else f"⚠️ Found {dupes} duplicates!")


In [ ]:
# ─────────────────────────────────────────────────
# LOOK FOR HIDDEN ISSUES: -9999 SENTINEL VALUES
# Some datasets use -9999 to mean "missing"
# ─────────────────────────────────────────────────
for col in ['u','g','r','i','z']:
    n = (df[col] < -100).sum()
    if n > 0:
        print(f"⚠️  Column '{col}' has {n:,} values below -100 (likely sentinel/error values)")
    else:
        print(f"✅  Column '{col}' looks clean")


## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────────
# EDA 1: REDSHIFT DISTRIBUTION BY CLASS
# WHY: Redshift is the most important feature.
# STARs have ~0, GALAXYs have moderate, QSOs have very high.
# ─────────────────────────────────────────────────
COLORS = {'GALAXY': '#3498db', 'STAR': '#f39c12', 'QSO': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in ['GALAXY', 'STAR', 'QSO']:
    data = df[df['class'] == cls]['redshift'].clip(-0.5, 7)
    axes[0].hist(data, bins=80, alpha=0.6, label=cls, color=COLORS[cls], density=True)
axes[0].set_title('Redshift Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Redshift Value')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
data_per_class = [df[df['class']==c]['redshift'].clip(-0.5,7) for c in ['GALAXY','STAR','QSO']]
axes[1].boxplot(data_per_class, labels=['GALAXY','STAR','QSO'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Redshift Boxplot per Class', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Redshift')

plt.suptitle('Redshift Analysis — Most Powerful Feature!', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("Key Insight:")
for cls in ['GALAXY','STAR','QSO']:
    mean_rs = df[df['class']==cls]['redshift'].mean()
    print(f"  {cls}: mean redshift = {mean_rs:.3f}")


In [ ]:
# ─────────────────────────────────────────────────
# EDA 2: PHOTOMETRIC BANDS BY CLASS
# WHY: Different object types emit different amounts
# of light in UV, Green, Red, IR bands.
# ─────────────────────────────────────────────────
bands = ['u', 'g', 'r', 'i', 'z']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, band in enumerate(bands):
    for cls in ['GALAXY', 'STAR', 'QSO']:
        # Filter out sentinel values before plotting
        data = df[df['class']==cls][band]
        data = data[(data > -100) & (data < 35)]
        axes[idx].hist(data, bins=60, alpha=0.6, label=cls,
                       color=COLORS[cls], density=True)
    axes[idx].set_title(f'Band: {band.upper()}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Magnitude (higher = fainter)')
    axes[idx].legend()

axes[5].set_visible(False)  # Hide unused subplot
plt.suptitle('Photometric Band Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# EDA 3: CORRELATION HEATMAP
# WHY: Understand which features are related to
# each other — helps avoid redundancy.
# ─────────────────────────────────────────────────
numeric_cols = ['alpha','delta','u','g','r','i','z','redshift']

# Filter out bad values for correlation
df_corr = df[numeric_cols].copy()
for col in ['u','g','r','i','z']:
    df_corr = df_corr[df_corr[col] > -100]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Show only lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("Key Insight: u, g, r, i, z bands are highly correlated (>0.9)")
print("This means they measure similar things — differences (color indices) are more useful!")


In [ ]:
# ─────────────────────────────────────────────────
# EDA 4: SKY POSITION SCATTER PLOT
# WHY: See if position in sky helps classify objects
# ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

for cls in ['GALAXY', 'STAR', 'QSO']:
    sample = df[df['class']==cls].sample(2000, random_state=42)
    ax.scatter(sample['alpha'], sample['delta'],
               alpha=0.2, s=5, label=cls, color=COLORS[cls])

ax.set_title('Sky Position: Right Ascension vs Declination', fontsize=13, fontweight='bold')
ax.set_xlabel('Right Ascension (alpha)'); ax.set_ylabel('Declination (delta)')
ax.legend(markerscale=4)
plt.tight_layout(); plt.show()

print("Key Insight: Sky position alone doesn't separate classes well — all mixed together")
print("This confirms redshift + photometric bands are more important.")


## Step 5: Data Cleaning

In [ ]:
# ─────────────────────────────────────────────────
# STEP 1: DROP USELESS COLUMNS
# WHY: These are just administrative IDs and don't
# carry information about WHAT the object is.
# ─────────────────────────────────────────────────
id_columns = [
    'obj_ID',       # Just a catalog identifier
    'spec_obj_ID',  # Spectroscopic catalog ID
    'run_ID',       # Scan run number
    'rerun_ID',     # Reprocessing ID
    'cam_col',      # Camera column
    'field_ID',     # Field number
    'plate',        # Plate number
    'MJD',          # Date of observation
    'fiber_ID',     # Fiber number
]
df_clean = df.drop(columns=id_columns)
print(f"✅ Dropped {len(id_columns)} columns.")
print(f"   Shape before: {df.shape} → after: {df_clean.shape}")
print(f"   Remaining columns: {df_clean.columns.tolist()}")


In [ ]:
# ─────────────────────────────────────────────────
# STEP 2: HANDLE -9999 SENTINEL VALUES
# WHY: Some observations have -9999 as a placeholder
# for "measurement failed". We replace with median.
# ─────────────────────────────────────────────────
for band in ['u', 'g', 'r', 'i', 'z']:
    # Compute median from valid values only (not -9999)
    valid_median = df_clean[df_clean[band] > -100][band].median()
    bad_count = (df_clean[band] < -100).sum()
    
    # Replace bad values with median
    df_clean[band] = df_clean[band].apply(lambda x: valid_median if x < -100 else x)
    
    if bad_count > 0:
        print(f"  Band '{band}': replaced {bad_count:,} bad values with median={valid_median:.2f}")
    else:
        print(f"  Band '{band}': No bad values found ✅")


## Step 6: Outlier Handling

In [ ]:
# ─────────────────────────────────────────────────
# DETECT OUTLIERS — BOXPLOT VISUALIZATION
# WHY: Outliers can distort model training, especially
# for distance-based models (KNN, SVM, Logistic Regression)
# ─────────────────────────────────────────────────
features_to_check = ['u', 'g', 'r', 'i', 'z', 'redshift']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, feat in enumerate(features_to_check):
    axes[idx].boxplot(df_clean[feat].dropna(), vert=True, patch_artist=True,
                      boxprops=dict(facecolor='lightcoral'),
                      medianprops=dict(color='black', linewidth=2))
    axes[idx].set_title(f'{feat.upper()}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Value')

plt.suptitle('Boxplots — Outlier Detection (Before Capping)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# HANDLE OUTLIERS — IQR CAPPING (WINSORIZATION)
# WHY: We cap extreme values instead of deleting rows.
# Capping preserves data while removing extreme distortion.
# We do NOT cap redshift — QSOs legitimately have high values!
# ─────────────────────────────────────────────────
def cap_outliers_iqr(df, column, factor=3.0):
    """
    Cap outliers using IQR method.
    Values beyond [Q1 - factor*IQR, Q3 + factor*IQR] are clipped.
    factor=3.0 is conservative — only removes extreme outliers.
    """
    Q1 = df[column].quantile(0.25)   # 25th percentile
    Q3 = df[column].quantile(0.75)   # 75th percentile
    IQR = Q3 - Q1                    # Interquartile Range
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    n_capped = ((df[column] < lower) | (df[column] > upper)).sum()
    df[column] = df[column].clip(lower=lower, upper=upper)
    return n_capped

# Apply to photometric bands ONLY (not redshift)
total = 0
for band in ['u', 'g', 'r', 'i', 'z']:
    n = cap_outliers_iqr(df_clean, band, factor=3.0)
    total += n
    print(f"  Band '{band}': {n:,} values capped")

print(f"\n✅ Total values capped: {total:,}")
print("   Note: redshift was NOT capped — high values are real QSOs!")


## Step 7: Feature Engineering

In [ ]:
# ─────────────────────────────────────────────────
# CREATE COLOR INDEX FEATURES
# WHY: The DIFFERENCE between adjacent photometric bands
# captures the "color" or spectral shape of an object.
# This is a standard technique in astronomy.
# QSOs have very different color indices from Stars and Galaxies.
# ─────────────────────────────────────────────────
df_clean['u_g'] = df_clean['u'] - df_clean['g']   # UV - Green
df_clean['g_r'] = df_clean['g'] - df_clean['r']   # Green - Red
df_clean['r_i'] = df_clean['r'] - df_clean['i']   # Red - Near-IR
df_clean['i_z'] = df_clean['i'] - df_clean['z']   # Near-IR - IR

print("✅ Created 4 new color index features: u_g, g_r, r_i, i_z")
print(f"   New shape: {df_clean.shape}")
print(f"   Features: {df_clean.columns.tolist()}")


## Step 8: Data Preprocessing (Encoding + Split + Scale)

In [ ]:
# ─────────────────────────────────────────────────
# LABEL ENCODING
# WHY: ML models need numbers, not text strings.
# LabelEncoder converts: GALAXY→0, QSO→1, STAR→2
# ─────────────────────────────────────────────────
le = LabelEncoder()
df_clean['target'] = le.fit_transform(df_clean['class'])

print("Label Encoding Result:")
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} → {code}")

class_names = le.classes_  # Save for later use in plots


In [ ]:
# ─────────────────────────────────────────────────
# DEFINE FEATURES (X) AND TARGET (y)
# ─────────────────────────────────────────────────
feature_cols = [c for c in df_clean.columns if c not in ['class', 'target']]

X = df_clean[feature_cols]   # Input features
y = df_clean['target']       # Target labels

print(f"Features (X): {feature_cols}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"\nClass distribution in y: {dict(zip(*np.unique(y, return_counts=True)))}")


In [ ]:
# ─────────────────────────────────────────────────
# TRAIN-TEST SPLIT
# WHY: We train on 80% and test on 20% UNSEEN data.
# 'stratify=y' ensures equal class proportions in both sets.
# This prevents overfitting and gives honest accuracy scores.
# ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 20% for testing
    random_state=42,        # For reproducibility
    stratify=y              # Keep class proportions equal
)

print(f"Training set : {X_train.shape} ({len(X_train):,} samples)")
print(f"Test set     : {X_test.shape} ({len(X_test):,} samples)")
print()
print("Class distribution in Training set:")
for cls, cnt in zip(class_names, np.bincount(y_train)):
    print(f"  {cls}: {cnt:,} ({100*cnt/len(y_train):.1f}%)")


In [ ]:
# ─────────────────────────────────────────────────
# FEATURE SCALING — RobustScaler
# WHY: Distance-based models (KNN, SVM, Logistic Regression, ANN)
# are sensitive to feature scale. If 'alpha' ranges 0-360 and
# 'redshift' ranges 0-7, the model unfairly favors alpha.
#
# RobustScaler uses median and IQR (not mean/std).
# It is BETTER than StandardScaler when outliers exist.
#
# IMPORTANT: Fit ONLY on training data, then transform both sets.
# Fitting on test data = data leakage!
#
# Note: Tree-based models (DT, RF, XGBoost) don't need scaling.
# ─────────────────────────────────────────────────
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Learn from train + transform
X_test_scaled  = scaler.transform(X_test)        # Only transform (don't relearn)

print("✅ Features scaled using RobustScaler")
print(f"   Original 'redshift' range: {X_train['redshift'].min():.2f} to {X_train['redshift'].max():.2f}")
print(f"   Scaled  'redshift' range: {X_train_scaled[:,feature_cols.index('redshift')].min():.2f} to {X_train_scaled[:,feature_cols.index('redshift')].max():.2f}")


## Step 9: Feature Importance (Before Training)

In [ ]:
# ─────────────────────────────────────────────────
# FEATURE IMPORTANCE USING RANDOM FOREST
# WHY: Tells us which features matter most for classification.
# Helps understand the data and potentially remove useless features.
# ─────────────────────────────────────────────────
rf_for_importance = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_for_importance.fit(X_train, y_train)  # No scaling needed for trees

importance = pd.Series(rf_for_importance.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=False)

print("Feature Importance Rankings:")
for feat, imp in importance.items():
    bar = '█' * int(imp * 100)
    print(f"  {feat:<12}: {bar} {imp:.4f} ({imp*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 6))
importance.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance Score')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

print(f"\n🥇 Most important feature: '{importance.index[0]}' ({importance.iloc[0]*100:.1f}%)")
print("   This explains why even simple models like Logistic Regression achieve >95%!")


## Step 10: Machine Learning Models

We will now train 12 different algorithms. For each one:
- **Why** we use it
- **How** it works (simple explanation)
- **Code** to train it
- **Results** on test data

We store all results in a dictionary for final comparison.


In [ ]:
# Helper function to evaluate any model
results = {}   # Store all results here

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train and evaluate a model. Prints results and saves to 'results' dict."""
    # Train the model
    model.fit(X_tr, y_tr)
    
    # Predict on unseen test data
    y_pred = model.predict(X_te)
    
    # Calculate metrics
    acc = accuracy_score(y_te, y_pred)           # Overall accuracy
    f1  = f1_score(y_te, y_pred, average='weighted')  # F1 (handles class imbalance)
    
    # Store results
    results[name] = {'Accuracy': round(acc*100, 2), 'F1_Score': round(f1, 4)}
    
    # Print results
    print(f"{'='*55}")
    print(f"  Model : {name}")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  Classification Report:")
    print(classification_report(y_te, y_pred, target_names=class_names))
    
    # Confusion matrix
    fig, ax = plt.subplots(figsize=(7, 5))
    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    
    return model


### Model 1: Logistic Regression

In [ ]:
# ─────────────────────────────────────────────────
# LOGISTIC REGRESSION
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Great baseline model — always start here
#   - Fast, interpretable, gives probability estimates
#   - Works well when features have linear relationships with class
#
# HOW IT WORKS:
#   - Finds a hyperplane (straight decision boundary) that best separates classes
#   - Uses sigmoid function to convert scores into probabilities
#   - For multi-class, uses 'one-vs-rest' approach (3 binary classifiers)
#
# NEEDS SCALING: Yes (uses gradient descent which needs equal-scale features)
# ─────────────────────────────────────────────────
lr_model = LogisticRegression(
    max_iter=1000,    # Max iterations for convergence
    random_state=42,  # Reproducibility
    n_jobs=-1         # Use all CPU cores
)
evaluate_model("Logistic Regression", lr_model, X_train_scaled, X_test_scaled, y_train, y_test)


### Model 2: Naive Bayes

In [ ]:
# ─────────────────────────────────────────────────
# NAIVE BAYES (Gaussian)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Very fast — trains in milliseconds
#   - Works surprisingly well as a baseline
#   - Good when features are roughly independent
#
# HOW IT WORKS:
#   - Applies Bayes Theorem: P(class | features) ∝ P(features | class) × P(class)
#   - 'Naive' = assumes features are INDEPENDENT of each other
#   - Gaussian = assumes each feature follows a normal distribution per class
#   - Predicts class with highest probability
#
# NEEDS SCALING: No (but doesn't hurt)
# ─────────────────────────────────────────────────
nb_model = GaussianNB()   # No hyperparameters needed!
evaluate_model("Naive Bayes", nb_model, X_train_scaled, X_test_scaled, y_train, y_test)


### Model 3: Support Vector Machine (SVM)

In [ ]:
# ─────────────────────────────────────────────────
# SUPPORT VECTOR MACHINE (LinearSVC)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Excellent for high-dimensional data
#   - Robust and works well with clear class separation
#   - LinearSVC is fast for large datasets (100k rows)
#
# HOW IT WORKS:
#   - Finds the hyperplane that MAXIMIZES the margin between classes
#   - 'Support Vectors' = data points closest to the boundary
#   - A wider margin → better generalization to new data
#   - LinearSVC = linear kernel (straight-line boundary)
#
# NEEDS SCALING: Yes (maximizing margin requires same-scale features)
# ─────────────────────────────────────────────────
svm_model = LinearSVC(
    max_iter=3000,   # More iterations for convergence
    random_state=42
)
evaluate_model("SVM (LinearSVC)", svm_model, X_train_scaled, X_test_scaled, y_train, y_test)


### Model 4: K-Nearest Neighbors (KNN)

In [ ]:
# ─────────────────────────────────────────────────
# K-NEAREST NEIGHBORS (KNN)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - No training phase — simple and intuitive
#   - Naturally handles multi-class
#   - Good when similar objects have similar features
#
# HOW IT WORKS:
#   - For each test point, find K=5 nearest training points
#   - 'Nearest' = smallest Euclidean distance in feature space
#   - Predict the most common class among those 5 neighbors
#   - Think of it as: "Tell me your neighbors and I'll tell you who you are"
#
# NEEDS SCALING: YES! (distance is meaningless if features have different scales)
# ─────────────────────────────────────────────────
knn_model = KNeighborsClassifier(
    n_neighbors=5,  # K=5: use 5 nearest neighbors
    n_jobs=-1       # Use all CPU cores for fast distance computation
)
evaluate_model("KNN (k=5)", knn_model, X_train_scaled, X_test_scaled, y_train, y_test)


### Model 5: Decision Tree

In [ ]:
# ─────────────────────────────────────────────────
# DECISION TREE
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Most interpretable model — you can read the rules
#   - No scaling needed
#   - Foundation of all tree-based ensemble methods
#
# HOW IT WORKS:
#   - At each node, picks the feature + threshold that best splits the data
#   - "Best" = minimizes Gini Impurity (measures how mixed classes are)
#   - Example rule: IF redshift < 0.01 → STAR; ELIF g_r < 0.3 → QSO; ELSE → GALAXY
#   - Grows until max_depth=15 (prevents overfitting)
#
# NEEDS SCALING: No (thresholds are relative to each feature)
# ─────────────────────────────────────────────────
dt_model = DecisionTreeClassifier(
    max_depth=15,     # Limit tree depth to prevent overfitting
    random_state=42
)
evaluate_model("Decision Tree", dt_model, X_train, X_test, y_train, y_test)


### Model 6: Random Forest (Ensemble — Bagging)

In [ ]:
# ─────────────────────────────────────────────────
# RANDOM FOREST
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Much more accurate than a single Decision Tree
#   - Resistant to overfitting
#   - Gives feature importance scores
#   - One of the best models for tabular data
#
# HOW IT WORKS:
#   - Builds 100 different Decision Trees
#   - Each tree trains on a DIFFERENT random sample (Bootstrap sampling)
#   - Each split considers a RANDOM subset of features
#   - Final prediction = majority vote across all 100 trees
#   - "Wisdom of the crowd" principle — many imperfect trees → one great model
#
# NEEDS SCALING: No
# ─────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,  # Number of trees to build
    random_state=42,
    n_jobs=-1          # Parallelize tree building
)
evaluate_model("Random Forest", rf_model, X_train, X_test, y_train, y_test)


### Model 7: Extra Trees

In [ ]:
# ─────────────────────────────────────────────────
# EXTRA TREES (Extremely Randomized Trees)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Faster than Random Forest
#   - More randomness → can reduce variance further
#
# HOW IT WORKS:
#   - Like Random Forest, but uses COMPLETELY RANDOM thresholds at each split
#   - Random Forest: tries multiple thresholds, picks BEST
#   - Extra Trees: picks a RANDOM threshold → much faster
#   - More randomness can help reduce overfitting
#
# NEEDS SCALING: No
# ─────────────────────────────────────────────────
et_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
evaluate_model("Extra Trees", et_model, X_train, X_test, y_train, y_test)


### Model 8: AdaBoost (Ensemble — Boosting)

In [ ]:
# ─────────────────────────────────────────────────
# ADABOOST (Adaptive Boosting)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Good example of boosting concept
#   - Focuses on hard-to-classify samples
#
# HOW IT WORKS:
#   - Trains weak learners (shallow trees, depth=1) SEQUENTIALLY
#   - After each tree, samples that were MISCLASSIFIED get higher weights
#   - Next tree tries harder to correctly classify those difficult samples
#   - Final prediction = weighted vote of all weak learners
#   - Note: Lower accuracy here because depth-1 stumps are too simple
#
# NEEDS SCALING: No
# ─────────────────────────────────────────────────
ada_model = AdaBoostClassifier(
    n_estimators=100,   # Number of weak learners
    random_state=42
)
evaluate_model("AdaBoost", ada_model, X_train, X_test, y_train, y_test)


### Model 9: Bagging Classifier

In [ ]:
# ─────────────────────────────────────────────────
# BAGGING (Bootstrap Aggregating)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Reduces variance (overfitting) of any base model
#   - General version of Random Forest
#
# HOW IT WORKS:
#   - Creates N bootstrap samples (random samples WITH replacement)
#   - Trains one Decision Tree on each sample
#   - Final prediction = majority vote
#   - Random Forest = Bagging + random feature subsets at each split
#
# NEEDS SCALING: No (uses trees as base)
# ─────────────────────────────────────────────────
bag_model = BaggingClassifier(
    n_estimators=25,   # Number of base models
    random_state=42,
    n_jobs=-1
)
evaluate_model("Bagging", bag_model, X_train, X_test, y_train, y_test)


### Model 10: XGBoost

In [ ]:
# ─────────────────────────────────────────────────
# XGBOOST (Extreme Gradient Boosting)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Wins most Kaggle competitions on tabular data
#   - Handles missing values automatically
#   - Built-in regularization to prevent overfitting
#
# HOW IT WORKS:
#   - Builds trees SEQUENTIALLY (like AdaBoost)
#   - But uses GRADIENT DESCENT on the loss function
#   - Each new tree tries to correct RESIDUAL ERRORS of previous trees
#   - learning_rate=0.1: each tree contributes 10% to reduce overfitting
#   - Much better than AdaBoost because trees can be deeper
#
# NEEDS SCALING: No
# ─────────────────────────────────────────────────
xgb_model = XGBClassifier(
    n_estimators=80,           # Number of trees
    max_depth=6,               # Maximum depth per tree
    learning_rate=0.1,         # Step size (shrinkage)
    eval_metric='mlogloss',    # Multi-class log loss
    random_state=42,
    n_jobs=-1,
    tree_method='hist'         # Faster histogram-based algorithm
)
evaluate_model("XGBoost", xgb_model, X_train, X_test, y_train, y_test)


### Model 11: LightGBM

In [ ]:
# ─────────────────────────────────────────────────
# LIGHTGBM (Microsoft's Light Gradient Boosting)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Fastest gradient boosting algorithm
#   - Excellent on large datasets (100k rows)
#   - Similar or better accuracy than XGBoost
#
# HOW IT WORKS:
#   - Like XGBoost but grows trees LEAF-WISE (not level-wise)
#   - Level-wise: all leaves at same depth expanded together
#   - Leaf-wise: always expand the leaf with maximum gain
#   - Uses histogram-based algorithm + GOSS sampling = very fast
#
# NEEDS SCALING: No
# ─────────────────────────────────────────────────
lgb_model = LGBMClassifier(
    n_estimators=80,
    random_state=42,
    n_jobs=-1,
    verbose=-1       # Suppress training logs
)
evaluate_model("LightGBM", lgb_model, X_train, X_test, y_train, y_test)


### Model 12: ANN — Artificial Neural Network (MLP)

In [ ]:
# ─────────────────────────────────────────────────
# ARTIFICIAL NEURAL NETWORK (MLP — Multi-Layer Perceptron)
# ─────────────────────────────────────────────────
# WHY USE IT:
#   - Learns complex non-linear patterns
#   - Universal approximator — can learn any function
#   - Foundation of deep learning
#
# HOW IT WORKS:
#   Input Layer (12 features)
#     ↓
#   Hidden Layer 1: 128 neurons (ReLU activation)
#     ↓
#   Hidden Layer 2: 64 neurons (ReLU activation)
#     ↓
#   Output Layer: 3 neurons (Softmax → probabilities)
#
#   Training: Backpropagation + Adam optimizer
#   Each epoch, weights are updated to reduce prediction error
#   early_stopping: stops training if validation loss stops improving
#
# NEEDS SCALING: YES! (gradients explode/vanish with different scales)
# ─────────────────────────────────────────────────
ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    activation='relu',              # ReLU activation function
    solver='adam',                  # Adam optimizer
    max_iter=100,                   # Maximum training epochs
    random_state=42,
    early_stopping=True,            # Stop if no improvement
    validation_fraction=0.1         # 10% of train for validation
)
evaluate_model("ANN (MLP)", ann_model, X_train_scaled, X_test_scaled, y_train, y_test)


## Step 11: Final Model Comparison

In [ ]:
# ─────────────────────────────────────────────────
# COMPARE ALL MODELS
# ─────────────────────────────────────────────────
# Create a summary DataFrame
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
comparison_df.index.name = 'Model'

print("\n🏆 FINAL MODEL LEADERBOARD")
print("="*50)
print(comparison_df.to_string())
print("="*50)
print(f"\n🥇 Best Model : {comparison_df.index[0]} ({comparison_df['Accuracy'].iloc[0]}%)")
print(f"🥉 Worst Model: {comparison_df.index[-1]} ({comparison_df['Accuracy'].iloc[-1]}%)")


In [ ]:
# ─────────────────────────────────────────────────
# VISUALIZATION: MODEL COMPARISON CHART
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Color code by performance
bar_colors = ['#2ecc71' if a >= 97 else '#f39c12' if a >= 94 else '#e74c3c'
              for a in comparison_df['Accuracy']]

# Accuracy chart
bars = axes[0].barh(comparison_df.index, comparison_df['Accuracy'],
                     color=bar_colors, edgecolor='black', height=0.6)
axes[0].set_xlim(60, 104)
axes[0].set_title('Model Accuracy (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Accuracy (%)')
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{w:.2f}%', va='center', fontweight='bold', fontsize=10)

# F1 Score chart
axes[1].barh(comparison_df.index, comparison_df['F1_Score'],
             color='steelblue', edgecolor='black', height=0.6)
axes[1].set_xlim(0.6, 1.05)
axes[1].set_title('F1 Score (Weighted)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('F1 Score')

plt.suptitle('All Models — Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# FINAL CONCLUSIONS
# ─────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║           KEY TAKEAWAYS FROM THIS PROJECT                ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  1. BEST FEATURE: 'redshift' (50.5% importance)          ║
║     → QSOs have high redshift, Stars have ~0             ║
║                                                          ║
║  2. BEST MODELS: Bagging ≈ LightGBM ≈ Random Forest     ║
║     → Ensemble tree models dominate tabular data         ║
║                                                          ║
║  3. WEAKEST: AdaBoost                                    ║
║     → Shallow stumps (depth=1) too simple for this task  ║
║                                                          ║
║  4. ALL MODELS > 92%                                     ║
║     → Strong features make the job easier                ║
║                                                          ║
║  5. FEATURE ENGINEERING HELPED                           ║
║     → Color indices (u-g, g-r, etc.) added value         ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")
